# 02 - Prétraitement des données

## Objectif

Ce notebook prépare les données issues du Data Warehouse pour leur utilisation dans les modèles de Machine Learning.

Les opérations réalisées sont :
- La sélection des variables pertinentes.
- La vérification de la qualité des données.
- La définition de la variable cible.
- La séparation chronologique des données en entraînement, validation et test.
- La préparation des variables numériques et catégorielles.

La variable cible est `is_fraud`.

La variable `is_flagged_fraud` est conservée pour les analyses comparatives, mais exclue des variables explicatives.

Importation des bibliothèques

In [62]:
import os
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

Connexion au Data Warehouse Render

In [63]:
# Localiser le fichier .env
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / ".env").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# Récupérer l'URL de connexion
database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise ValueError("DATABASE_URL introuvable dans le fichier .env.")

# Adapter l'URL au pilote PostgreSQL
if database_url.startswith("postgres://"):
    database_url = database_url.replace(
        "postgres://", "postgresql+psycopg2://", 1
    )
elif database_url.startswith("postgresql://"):
    database_url = database_url.replace(
        "postgresql://", "postgresql+psycopg2://", 1
    )

# Créer le moteur de connexion
engine = create_engine(
    database_url,
    connect_args={"sslmode": "require"},
    pool_pre_ping=True
)

# Vérifier la connexion
with engine.connect() as connection:
    version = connection.execute(text("SELECT version()")).scalar()

print("Connexion à Render réussie !")
print("Version PostgreSQL :", version)

Connexion à Render réussie !
Version PostgreSQL : PostgreSQL 18.7 (Debian 18.7-1.pgdg12+5) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


Extraction des données du DWH

In [64]:
query = """
SELECT
    f.transaction_id,
    d.step,
    d.jour,
    d.heure,
    d.tranche_horaire,
    d.type_jour,
    t.code_type AS transaction_type,
    t.categorie AS transaction_category,
    f.amount,
    orig.account_type AS orig_type,
    dest.account_type AS dest_type,
    f.is_fraud,
    f.is_flagged_fraud

FROM fact_transactions f

JOIN dim_type_operation t
    ON f.type_id = t.type_id

JOIN dim_temps d
    ON f.time_id = d.time_id

JOIN dim_compte_emetteur orig
    ON f.orig_account_id = orig.orig_account_id

JOIN dim_compte_destinataire dest
    ON f.dest_account_id = dest.dest_account_id;
"""

print("Extraction des données depuis Render...")

with engine.connect() as connection:
    df = pd.read_sql_query(text(query), connection)

print(f"Transactions récupérées : {len(df):,}")
print(f"Nombre de variables : {df.shape[1]}")

display(df.head())

Extraction des données depuis Render...
Transactions récupérées : 299,983
Nombre de variables : 13


,transaction_id,step,jour,heure,tranche_horaire,type_jour,transaction_type,transaction_category,amount,orig_type,dest_type,is_fraud,is_flagged_fraud
0,23,130,6,9,Matin,Week-end,CASH_OUT,Retrait,179718.53,Client,Client,0,0
1,109,16,1,15,Après-midi,Semaine,CASH_OUT,Retrait,243534.61,Client,Client,0,0
2,132,347,15,10,Matin,Semaine,CASH_IN,Dépôt,293881.02,Client,Client,0,0
3,163,275,12,10,Matin,Semaine,CASH_OUT,Retrait,114762.09,Client,Client,0,0
4,168,34,2,9,Matin,Semaine,PAYMENT,Achat,62175.13,Client,Commerçant,0,0


Sélection des variables

In [65]:
# Variables explicatives
features = [
    "transaction_type",
    "amount",
    "heure",
    "tranche_horaire",
    "type_jour",
    "orig_type",
    "dest_type"
]

# Variable cible
target = "is_fraud"

# Vérification des colonnes nécessaires
missing_columns = [
    col for col in features + [target, "step"]
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Colonnes manquantes : {missing_columns}")

# Construction du jeu de données
data = df[features + [target, "step", "is_flagged_fraud"]].copy()

print("Variables explicatives :", features)
print("Variable cible :", target)
print("Dimensions du jeu de données :", data.shape)

Variables explicatives : ['transaction_type', 'amount', 'heure', 'tranche_horaire', 'type_jour', 'orig_type', 'dest_type']
Variable cible : is_fraud
Dimensions du jeu de données : (299983, 10)


Vérification de la qualité des données

In [66]:
print("Valeurs manquantes :")
print(data.isnull().sum())

print("\nNombre de doublons :", data.duplicated().sum())

print("\nRépartition de la variable cible :")
print(data[target].value_counts())

print("\nMontants non positifs :", (data["amount"] <= 0).sum())

print("\nValeurs de la cible :")
print(sorted(data[target].unique()))

Valeurs manquantes :
transaction_type    0
amount              0
heure               0
tranche_horaire     0
type_jour           0
orig_type           0
dest_type           0
is_fraud            0
step                0
is_flagged_fraud    0
dtype: int64

Nombre de doublons : 247

Répartition de la variable cible :
is_fraud
0    291786
1      8197
Name: count, dtype: int64

Montants non positifs : 0

Valeurs de la cible :
[np.int64(0), np.int64(1)]


Séparation chronologique
Nous allons utiliser une répartition de 70 % / 15 % / 15 %.

Important : les pourcentages sont calculés sur les périodes distinctes et non directement sur les lignes. Cela évite de répartir les transactions d'une même période entre plusieurs ensembles.

In [67]:
# ============================================================
# 7. SÉPARATION CHRONOLOGIQUE DES DONNÉES
# ============================================================

# Trier les transactions par ordre chronologique
data = data.sort_values("step").reset_index(drop=True)

# Calculer le nombre de transactions pour chaque période
step_counts = data.groupby("step").size().sort_index()

# Calculer le nombre cumulé de transactions
cumulative_counts = step_counts.cumsum()

# Nombre total de transactions
total_transactions = len(data)

# Déterminer les limites correspondant à 70 % et 85 % des transactions
train_cut = (cumulative_counts - 0.70 * total_transactions).abs().idxmin()
valid_cut = (cumulative_counts - 0.85 * total_transactions).abs().idxmin()

# Créer les trois ensembles en respectant l'ordre chronologique
train = data[data["step"] <= train_cut].copy()

valid = data[
    (data["step"] > train_cut) &
    (data["step"] <= valid_cut)
].copy()

test = data[data["step"] > valid_cut].copy()

# Afficher les résultats
print("Séparation chronologique effectuée.")
print(f"Fin de l'entraînement : step {train_cut}")
print(f"Fin de la validation : step {valid_cut}")

print("\nRépartition des transactions :")
print(f"Entraînement : {len(train):,} ({len(train)/total_transactions*100:.2f} %)")
print(f"Validation   : {len(valid):,} ({len(valid)/total_transactions*100:.2f} %)")
print(f"Test         : {len(test):,} ({len(test)/total_transactions*100:.2f} %)")

Séparation chronologique effectuée.
Fin de l'entraînement : step 324
Fin de la validation : step 379

Répartition des transactions :
Entraînement : 209,871 (69.96 %)
Validation   : 45,896 (15.30 %)
Test         : 44,216 (14.74 %)


Vérification des ensembles
Cette cellule est essentielle compte tenu des anomalies temporelles constatées lors de notre audit.

In [68]:
def summarize_dataset(name, dataset):
    print(f"\n--- {name} ---")
    print(f"Transactions : {len(dataset):,}")
    print(f"Nombre de périodes : {dataset['step'].nunique()}")
    print(f"Step minimum : {dataset['step'].min()}")
    print(f"Step maximum : {dataset['step'].max()}")
    print(f"Fraudes : {dataset['is_fraud'].sum():,}")
    print(f"Transactions normales : {(dataset['is_fraud'] == 0).sum():,}")
    print(f"Taux de fraude : {dataset['is_fraud'].mean() * 100:.2f} %")

summarize_dataset("ENTRAÎNEMENT", train)
summarize_dataset("VALIDATION", valid)
summarize_dataset("TEST", test)


--- ENTRAÎNEMENT ---
Transactions : 209,871
Nombre de périodes : 324
Step minimum : 1
Step maximum : 324
Fraudes : 3,650
Transactions normales : 206,221
Taux de fraude : 1.74 %

--- VALIDATION ---
Transactions : 45,896
Nombre de périodes : 55
Step minimum : 325
Step maximum : 379
Fraudes : 570
Transactions normales : 45,326
Taux de fraude : 1.24 %

--- TEST ---
Transactions : 44,216
Nombre de périodes : 364
Step minimum : 380
Step maximum : 743
Fraudes : 3,977
Transactions normales : 40,239
Taux de fraude : 8.99 %


Séparation des variables explicatives et de la cible

In [69]:
# Variables explicatives
X_train = train[features]
X_valid = valid[features]
X_test = test[features]

# Variables cibles
y_train = train[target]
y_valid = valid[target]
y_test = test[target]

# Identification des types de variables
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Variables catégorielles :", categorical_features)
print("Variables numériques :", numerical_features)

Variables catégorielles : ['transaction_type', 'tranche_horaire', 'type_jour', 'orig_type', 'dest_type']
Variables numériques : ['amount', 'heure']


Construction du pipeline de prétraitement
Nous définissons ici les transformations qui seront appliquées aux données avant l'entraînement.

In [70]:
# Prétraitement des variables numériques
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prétraitement des variables catégorielles
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Regroupement des transformations
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Pipeline de prétraitement créé avec succès.")

Pipeline de prétraitement créé avec succès.


Sauvegarde des données préparées

In [71]:
from pathlib import Path

# Création du dossier de sauvegarde
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Sauvegarde des ensembles
train.to_parquet(output_dir / "train.parquet", index=False)
valid.to_parquet(output_dir / "valid.parquet", index=False)
test.to_parquet(output_dir / "test.parquet", index=False)

print("Les trois ensembles ont été sauvegardés avec succès.")

Les trois ensembles ont été sauvegardés avec succès.
